# Análisis de Datos Masivos con Machine Learning
## Caso 3: Discriminación y Seguridad Ciudadana — Encuesta de Hogares 2025 (INE Bolivia)

- **Curso**: Análisis de Datos Masivos con Machine Learning · Módulo 3 · Sesión 2
- **Alumno**: JOSE CHIPANA  ·  `josschipanaensec24@gmail.com`
- **Fecha**: septiembre 2026
- **Plataforma en línea**: Google Colab
- **Datos**: ENCUESTA DE HOGARES 2025 — INE Bolivia (catálogo ANDA: `BOL-INE-EH-2025`)
- **Metodología**: CRISP-DM (Comprensión del negocio y los datos → Preparación → Modelado → Evaluación → Despliegue)

---

## Fase 0 · Comprensión del problema (CRISP-DM: Business & Data Understanding)

**Problema / objetivo de negocio:** La **discriminación** y la **percepción de inseguridad** son fenómenos
sociales que afectan la convivencia y el ejercicio de derechos. Bolivia cuenta con la **Ley N° 045 contra el
racismo y toda forma de discriminación**. Este trabajo usa los datos de la Encuesta de Hogares **2025** para:

1. Cuantificar la prevalencia de la discriminación y sus **motivos** más frecuentes.
2. Analizar la **victimización** y la **percepción de inseguridad**.
3. Construir **modelos de Machine Learning** para predecir `sufrió discriminación` a partir de
   características sociodemográficas, económicas y de la vivienda.
4. Identificar las **variables más influyentes** (importancia) y derivar **recomendaciones de política pública**.

**Módulos de la EH2025 utilizados:**

| Archivo | Módulo | Contenido |
|---|---|---|
| `EH2025_Persona.sav` | Características de las personas | Sexo, edad, lengua, autoidentificación indígena, educación, empleo, pobreza |
| `EH2025_Vivienda_1.sav` | Vivienda y hogar | Materiales, agua, saneamiento, electricidad, internet, hacinamiento |
| `EH2025_Discriminacion.sav` | **Módulo 9** | Discriminación, victimización, seguridad, confianza en la Policía |

**Variables objetivo (Módulo 9):** `s09a_01a`–`l` (sufrió discriminación por 12 motivos),
`s09b_02a/b` (victimización), `s09b_01` (seguridad caminando de noche), `s09a_02` (denuncia formal).

**Fuente:** Instituto Nacional de Estadística, Encuesta de Hogares 2025 - http://anda.ine.gob.bo/index.php/catalog/256. Fecha de acceso: septiembre de 2026.


In [ ]:
# @title 1. Configuración del entorno (instala y carga librerías)
import os, sys, warnings, subprocess, json, math
warnings.filterwarnings("ignore")

def pip_quiet(pkg):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

necesarios = {
 "pyreadstat": "pyreadstat", "matplotlib": "matplotlib", "seaborn": "seaborn",
 "folium": "folium", "plotly": "plotly", "kaleido": "kaleido", "sklearn": "scikit-learn",
 "shap": "shap", "reportlab": "reportlab"}
for mod, pkg in necesarios.items():
    try:
        __import__(mod)
    except Exception:
        pip_quiet(pkg)
        __import__(mod)

import pyreadstat, pandas as pd, numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import plotly.express as px
import plotly.io as pio
import plotly
import folium, shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, classification_report)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from google.colab import files
from IPython.display import display, HTML

pd.set_option("display.max_columns", 40)
print("Entorno listo. Versiones:")
print(" pandas", pd.__version__, "| numpy", np.__version__, "| sklearn", __import__("sklearn").__version__,
      "| folium", folium.__version__, "| plotly", plotly.__version__)


In [ ]:
# @title 2. Localización / carga de los datos .sav de la EH2025
# En Colab: si no se detectan los archivos, aparece un cuadro para subir los .sav
CANDIDATOS = ["/content", "data/raw", "data", "BD_EH2025", "..", "."]

def localizar(nombre):
    for d in CANDIDATOS:
        p = os.path.join(d, nombre)
        if os.path.exists(p):
            return p
    return None

DATA_DIR = None
for d in CANDIDATOS:
    if os.path.isdir(d) and any(f.endswith(".sav") for f in os.listdir(d)):
        DATA_DIR = d
        break

if DATA_DIR is None:
    print("Sube los archivos .sav de la EH2025 (Persona, Vivienda_1 y Discriminacion):")
    uploaded = files.upload()
    DATA_DIR = "/content"
print("Directorio de datos:", DATA_DIR)


In [ ]:
# @title 3. Carga de los tres módulos (Persona, Vivienda, Discriminación)
def leer(nombre):
    p = localizar(nombre)
    if p is None:
        p = os.path.join(DATA_DIR, nombre)
    df, meta = pyreadstat.read_sav(p)
    return df, meta

pers, meta_p = leer("EH2025_Persona.sav")
viv,  meta_v = leer("EH2025_Vivienda_1.sav")
disc, meta_d = leer("EH2025_Discriminacion.sav")

print("Persona        :", pers.shape)
print("Vivienda_1     :", viv.shape)
print("Discriminacion :", disc.shape)
print()
print("¿La Discriminación es un subconjunto de Persona?")
print("  Personas en Disc en Persona:",
      len(disc.set_index(["folio","nro"]).index.intersection(pers.set_index(["folio","nro"]).index)))
print("  Niveles de 'nro' (miembros) por hogar (Persona):", sorted(pers.nro.dropna().unique())[:15])


---

## Fase 1 · Obtención y preprocesamiento de datos (25%)

**Pasos:**
1. Diccionarios y recodificación (departamentos, área, sexo, etnia, lengua, educación, empleo).
2. Construcción de las **variables objetivo** (discriminación, victimización, inseguridad, denuncia).
3. Ingeniería de características de **vivienda** (materiales, servicios, hacinamiento).
4. **Fusión** de los tres módulos por `folio` / `folio + nro`.
5. Limpieza (valores perdidos), control de calidad y guardado de `dataset_procesado.csv`.


In [ ]:
# @title 4. Diccionarios y recodificación geográfica
DEPTOS = {1:"Chuquisaca",2:"La Paz",3:"Cochabamba",4:"Oruro",5:"Potosí",
          6:"Tarija",7:"Santa Cruz",8:"Beni",9:"Pando"}
AREA = {1:"Urbano", 2:"Rural"}
SEXO = {1:"Hombre", 2:"Mujer"}

MOTIVOS = {
 "s09a_01a":"Color de piel",
 "s09a_01b":"Nación pueblo indígena originario campesino o afroboliviano",
 "s09a_01c":"Procedencia regional o nacionalidad extranjera",
 "s09a_01d":"Orientación sexual e identidad de género",
 "s09a_01e":"Edad",
 "s09a_01f":"Sexo (hombre, mujer)",
 "s09a_01g":"Idioma",
 "s09a_01h":"Vestimenta",
 "s09a_01i":"Discapacidad",
 "s09a_01j":"Religión",
 "s09a_01k":"Condición económica o social",
 "s09a_01l":"Otro motivo"}

# lengua materna -> grupo (códigos EH2025)
LENG_ORIG = (list(range(1,40)) + [92,93,94,95,96,97,100])

def grupo_lengua(x):
    if pd.isna(x): return None
    x = int(x)
    if x in (995, 996, 999): return None      # no puede/no habla aún / no aplica
    if x == 6: return "Castellano"
    if x in LENG_ORIG: return "Originaria"
    return "Extranjera"


In [ ]:
# @title 5. Construcción de variables objetivo (Módulo 9: Discriminación y seguridad)
d = disc.copy()
d["depto"] = d["depto"].map(DEPTOS)
d["area"] = d["area"].map(AREA)

# --- TARGET 1: sufrió discriminación (al menos un motivo, Si = 1) ---
cols_motivo = [c for c in MOTIVOS.keys() if c in d.columns]
d["suf_disc"] = (d[cols_motivo] == 1).any(axis=1).astype(int)
# numero de motivos reportados
d["n_motivos"] = ((d[cols_motivo] == 1).sum(axis=1))

# --- TARGET 2: victimización (s09b_02a/b: 8 = ninguno) ---
d["victima"] = (d[["s09b_02a","s09b_02b"]].notna() & (d[["s09b_02a","s09b_02b"]] != 8)).any(axis=1).astype(int)

# --- TARGET 3: percepción de inseguridad (muy inseguro/inseguro) ---
d["inseguridad"] = (d["s09b_01"].le(2)).astype(int)

# --- TARGET 4: denuncia formal entre quienes sufrieron discriminación ---
d["denuncia"] = (d["s09a_02"] == 1).astype(int)

print("Prevalencias (muestra no expandida):")
for t in ["suf_disc","victima","inseguridad","denuncia"]:
    s = d[t]
    n = s.notna().sum()
    print(f"  {t:12s}  n={n:6d}  prevalencia={s.sum()/n*100:5.2f}%  (sí={s.sum()})")
print()
print("Desglose por motivo de discriminación:")
preval_motivo = ((d[cols_motivo] == 1).mean()*100).sort_values(ascending=False)
for c, v in preval_motivo.items():
    print(f"  {MOTIVOS[c]:48s} {v:5.2f}%")


In [ ]:
# @title 6. Características de Persona (predictores sociodemográficos y económicos)
p = pers.copy()
p["depto"] = p["depto"].map(DEPTOS)
p["area"] = p["area"].map(AREA)
p["sexo"] = p["s01a_02"].map(SEXO)
p["indigena"] = (p["s01a_09"] == 1).astype(int)          # se autoidentifica NPIOC o afroboliviano
p["lengua_grupo"] = p["s01a_08"].apply(grupo_lengua)
p["alfabetizado"] = (p["s03a_01"] == 1).astype(int)      # sabe leer y escribir
p["edad"] = p["s01a_03"].clip(0, 98)

# Nivel educativo -> años de escolaridad aproximados
ANIOS_ED = {0:0, 1:4, 2:6, 3:10, 4:12, 5:17, 6:12}
p["anios_educ"] = p["niv_ed"].map(ANIOS_ED)

# Condición de actividad: 1/4/5 ocupados, 2/3 desocupados
cond = p["condact"].copy()
p["ocupado"] = cond.isin([1,4,5]).astype(int)
p["desocupado"] = cond.isin([2,3]).astype(int)

# Ingresos (transformación log1p)
p["log_ylab"] = np.log1p(p["ylab"].replace(-9, np.nan))
p["log_yhogpc"] = np.log1p(p["yhogpc"].fillna(0))
p["pobre"] = (p["p0"] == 1).astype(int)
p["pobre_extremo"] = (p["pext0"] == 1).astype(int)

pred_pers = ["depto","area","sexo","edad","anios_educ","indigena","lengua_grupo",
             "alfabetizado","ocupado","desocupado","log_ylab","log_yhogpc",
             "pobre","pobre_extremo"]
print("Final Persona:", p.shape, "| variables:", pred_pers)
print("Rangos edad:", p["edad"].min(), "-", p["edad"].max())


In [ ]:
# @title 7. Características de Vivienda / hogar (predictores de contexto)
h = viv.drop_duplicates("folio").copy()

h["material_paredes_noble"] = (h["s06a_03"] == 1).astype(int)          # ladrillo/bloque/hormigón
h["piso_noble"] = h["s06a_06"].isin([3,4,6,7,8]).astype(int)            # parquet, baldosa, cerámica...
h["agua_red"] = h["s06a_07"].isin([1,2]).astype(int)                    # cañería de red (dentro/fuera)
h["saneamiento"] = (h["s06a_09"] == 1).astype(int)                      # baño con descarga de agua
h["electricidad"] = (h["s06a_12"] == 1).astype(int)
h["basura_recogida"] = h["s06a_13"].isin([5,6]).astype(int)             # basurero público / carro basurero
h["combustible_limpio"] = h["s06a_15"].isin([3,4,6]).astype(int)        # gas o electricidad
h["internet"] = (h["s06a_19"] == 1).astype(int)
h["vivienda_propia"] = h["s06a_02"].isin([1,2,5]).astype(int)
h["n_cuartos"] = h["s06a_16"]
h["n_dormitorios"] = h["s06a_17"]
h["hacinamiento"] = (h["totper"] / h["n_cuartos"].replace(0, np.nan)).round(2)

pred_hog = ["material_paredes_noble","piso_noble","agua_red","saneamiento","electricidad",
            "basura_recogida","combustible_limpio","internet","vivienda_propia",
            "n_cuartos","n_dormitorios","hacinamiento","totper"]
print("Final Vivienda (hogares únicos):", h.shape)
print("Hogares con hacinamiento > 3:", (h["hacinamiento"] > 3).sum())


In [ ]:
# @title 8. Fusión de los módulos y dataset final
# Unión Discriminación (nivel persona) + Vivienda (nivel hogar, por folio)
df = d.merge(h[["folio"] + pred_hog], on="folio", how="left", validate="m:1")
print("Tras unir Vivienda:", df.shape, "| hogares sin vivienda:", df[pred_hog[0]].isna().sum())

# 'depto' y 'area' ya vienen del módulo Discriminación; se excluyen del merge
_extra_p = ["factor","upm","estrato"]
p_merge = p[["folio","nro"] + [c for c in pred_pers if c not in ("depto","area")] + _extra_p]
df = df.merge(p_merge, on=["folio","nro"], how="left", validate="1:1")
print("Tras unir Persona:", df.shape)

print()
print("Cobertura del merge:")
print("  Persona con datos:", df["edad"].notna().mean()*100, "%")
# Variable objetivo principal
df["suf_disc"] = df["suf_disc"].astype(int)

# Guardar dataset procesado (para la fase de modelado y para GitHub)
df.to_csv("dataset_procesado.csv", index=False)
print()
print("Se guardó  dataset_procesado.csv  ->", df.shape)
print(df[["depto","area","edad","suf_disc","victima","inseguridad","denuncia",
         "log_yhogpc","pobre","hacinamiento","internet","factor"]].head(10).to_string())


In [ ]:
# @title 9. Control de calidad: datos faltantes y balance del target
print("Valores perdidos por bloque (%):")
na = pd.DataFrame({"% NA": df.isna().mean()*100})
print(na[na["% NA"] > 0].round(2).to_string())
print()
print("Balance de la variable objetivo 'suf_disc':")
print(df["suf_disc"].value_counts(normalize=True).round(4)*100)
print("   -> clase minoritaria ~", round(df["suf_disc"].mean()*100, 2), "% -> se usará class_weight='balanced'")


---

## Fase 2 · Análisis Exploratorio de Datos (EDA) — 4 visualizaciones (25%)

Se construyen **4 visualizaciones**:
1. **Mapa interactivo de Bolivia** (Folium): prevalencia de discriminación por departamento.
2. **Gráfico de barras**: % de discriminación por motivo.
3. **Relación**: discriminación y victimización según edad y sexo (y nivel socioeconómico).
4. **Clustering**: K-means + PCA sobre perfiles sociodemográficos (segmentación).


In [ ]:
# @title 10. Vista general del dataset procesado
print("Dimensiones:", df.shape)
print(df.dtypes.value_counts().to_string())
df[["edad","anios_educ","log_yhogpc","n_cuartos","hacinamiento","n_motivos"]].describe().T.round(2)


In [ ]:
# @title 11. Viz 1 · Mapa interactivo de Bolivia (Folium) — discriminación por departamento
geo = localizar("bolivia_departamentos.geojson")
resa = df.groupby("depto").apply(lambda g: pd.Series({
    "n": len(g),
    "tasa_disc": (g["suf_disc"].mean()*100).round(2),
    "tasa_victima": (g["victima"].mean()*100).round(2),
    "tasa_inseguridad": (g["inseguridad"].mean()*100).round(2)}), include_groups=False).reset_index()

mapa = folium.Map(location=[-16.5, -64.6], zoom_start=5, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=geo,
    name="Discriminación",
    data=resa, columns=["depto","tasa_disc"],
    key_on="feature.properties.shapeName",
    fill_color="YlOrRd", fill_opacity=0.7, line_opacity=0.6,
    legend_name="% personas que sufrieron discriminación (EH2025)").add_to(mapa)

tools = folium.features.GeoJson(geo, name="Tooltip",
    style_function=lambda x: {"fillColor":"transparent","color":"#555","weight":1.2})
for feature in tools.data["features"]:
    dep = feature["properties"]["shapeName"]
    row = resa[resa["depto"] == dep]
    if len(row):
        feature["properties"]["tasa"] = f"{row['tasa_disc'].iloc[0]:.2f}%"
        feature["properties"]["victima"] = f"{row['tasa_victima'].iloc[0]:.2f}%"
        feature["properties"]["inseg"] = f"{row['tasa_inseguridad'].iloc[0]:.2f}%"
tools.add_child(folium.features.GeoJsonTooltip(
    fields=["shapeName","tasa","victima","inseg"],
    aliases=["Departamento:","Discriminación:","Víctimas:","Inseguridad:"],
    localize=True, style="font-size:12px")).add_to(mapa)

# Capas adicionales (requisito Parte 2): ingreso per cápita e inseguridad por departamento
capa_ingreso = df.groupby("depto").apply(
    lambda g: pd.Series({"ingreso_medio": g["log_yhogpc"].mean().round(3)}), include_groups=False).reset_index()
capa_inseg = df.groupby("depto").apply(
    lambda g: pd.Series({"inseg": (g["inseguridad"].mean()*100).round(2)}), include_groups=False).reset_index()
folium.Choropleth(
    geo_data=geo, name="Ingreso per cápita (log)",
    data=capa_ingreso, columns=["depto", "ingreso_medio"],
    key_on="feature.properties.shapeName",
    fill_color="BuGn", fill_opacity=0.7, line_opacity=0.6,
    legend_name="Ingreso per cápita medio del hogar (log)").add_to(mapa)
folium.Choropleth(
    geo_data=geo, name="Inseguridad (%)",
    data=capa_inseg, columns=["depto", "inseg"],
    key_on="feature.properties.shapeName",
    fill_color="PuRd", fill_opacity=0.7, line_opacity=0.6,
    legend_name="% se siente inseguro de noche").add_to(mapa)
folium.LayerControl(collapsed=False).add_to(mapa)

mapa.save("mapa_discriminacion.html")
display(HTML('<div style="height:520px">'+mapa._repr_html_()+'</div>'))
print("Mapa interactivo guardado: mapa_discriminacion.html")
print(resa.round(2).to_string(index=False))


In [ ]:
# @title 12. Viz 2 · Gráfico de barras — % de discriminación por motivo
preval = ((df[list(MOTIVOS.keys())] == 1).mean()*100).sort_values()
labels = [MOTIVOS[c].replace("Nación pueblo indígena originario campesino o afroboliviano",
                              "Pertenencia a NPIOC o afroboliviano") for c in preval.index]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(labels, preval.values, color=sns.color_palette("Reds_r", len(preval)))
ax.bar_label(bars, fmt="%.2f%%", padding=3, fontsize=9)
ax.set_xlabel("% de personas que sufrieron discriminación en los últimos 12 meses")
ax.set_title("Frecuencia de motivos de discriminación — EH2025")
sns.despine(left=True)
plt.tight_layout()
plt.savefig("grafico_motivos_discriminacion.png", dpi=150)
plt.show()

# Interactivo (Plotly)
fig_px = px.bar(preval.sort_values(ascending=False).rename(index=lambda c: MOTIVOS[c]),
                orientation="h", labels={"value":"%", "index":""},
                title="Motivos de discriminación (interactivo — Plotly)")
fig_px.update_layout(height=500, margin=dict(l=20, r=20, t=60, b=20))
fig_px.show()


In [ ]:
# @title 13. Viz 3 · Relación — edad, sexo, condición socioeconómica vs discriminación/victimización
df2 = df.dropna(subset=["edad","sexo"]).copy()
df2["grupo_edad"] = pd.cut(df2["edad"], bins=[0,14,24,34,44,54,64,120],
                           labels=["0-14","15-24","25-34","35-44","45-54","55-64","65+"])

curva = df2.groupby(["grupo_edad","sexo"], observed=True).apply(
    lambda g: pd.Series({"disc": g["suf_disc"].mean()*100,
                         "vict": g["victima"].mean()*100}), include_groups=False).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
sns.lineplot(data=curva, x="grupo_edad", y="disc", hue="sexo", marker="o", ax=axes[0])
axes[0].set_title("Discriminación por grupo de edad y sexo")
axes[0].set_ylabel("% sufrió discriminación"); axes[0].set_xlabel("")
sns.lineplot(data=curva, x="grupo_edad", y="vict", hue="sexo", marker="o", ax=axes[1])
axes[1].set_title("Victimización por grupo de edad y sexo")
axes[1].set_ylabel("% fue víctima de delito"); axes[1].set_xlabel("")
for ax in axes: ax.grid(alpha=.3); sns.despine(ax=ax)
plt.tight_layout()
plt.savefig("grafico_edad_sexo.png", dpi=150)
plt.show()

# Relación con nivel socioeconómico
se = df.dropna(subset=["log_yhogpc","suf_disc"]).copy()
se["decil_yhogpc"] = pd.qcut(se["log_yhogpc"], 10, labels=False, duplicates="drop")
curva2 = se.groupby("decil_yhogpc").apply(lambda g: pd.Series({
    "disc": g["suf_disc"].mean()*100,
    "inseg": g["inseguridad"].mean()*100}), include_groups=False).reset_index()

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(curva2["decil_yhogpc"], curva2["disc"], marker="o", label="Discriminación")
ax.plot(curva2["decil_yhogpc"], curva2["inseg"], marker="s", label="Percepción de inseguridad")
ax.set_xlabel("Decil de ingreso per cápita del hogar (1 = más pobre)")
ax.set_ylabel("%")
ax.set_title("Discriminación e inseguridad por decil de ingreso")
ax.legend(); ax.grid(alpha=.3); sns.despine()
plt.tight_layout()
plt.savefig("grafico_ingreso.png", dpi=150)
plt.show()


In [ ]:
# @title 14. Viz 4 · Clustering K-means + PCA (segmentación de perfiles)
cols_cluster = ["edad","anios_educ","log_yhogpc","hacinamiento","pobre","indigena",
                "ocupado","desocupado","n_motivos","inseguridad"]
muestra = df[cols_cluster].dropna().sample(n=5000, random_state=7)
Xc = (muestra - muestra.mean()) / muestra.std()

inercias = []
for k in range(1, 9):
    km = KMeans(n_clusters=k, n_init=10, random_state=7)
    km.fit(Xc)
    inercias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.plot(range(1, 9), inercias, marker="o")
ax.set_xlabel("Número de clusters (k)"); ax.set_ylabel("Inercia")
ax.set_title("Método del codo para elegir k")
plt.savefig("grafico_codo.png", dpi=150)
plt.show()

k_final = 4
km = KMeans(n_clusters=k_final, n_init=10, random_state=7)
clusters = km.fit_predict(Xc)
muestra_et = muestra.copy()
muestra_et["cluster"] = clusters

pca = PCA(n_components=2, random_state=7)
comp = pca.fit_transform(Xc)
muestra_et["PC1"], muestra_et["PC2"] = comp[:, 0], comp[:, 1]

fig, ax = plt.subplots(figsize=(7.5, 5.2))
sc = ax.scatter(muestra_et["PC1"], muestra_et["PC2"], c=muestra_et["cluster"],
                cmap="viridis", s=18, alpha=0.8)
ax.set_xlabel(f"Componente principal 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"Componente principal 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("K-means (k=4) — perfiles sociodemográficos proyectados con PCA")
plt.colorbar(sc, label="Cluster")
plt.tight_layout()
plt.savefig("grafico_clustering.png", dpi=150)
plt.show()

print("Perfil promedio por cluster:")
perfil = muestra_et.groupby("cluster").mean(numeric_only=True).round(2)
print(perfil.to_string())

# --- Interpretación y etiquetado de cada cluster (requisito Parte 2) ---
ETIQUETAS = {
    0: "Adultos mayores indígenas, contexto vulnerable",
    1: "Población acomodada, escolarizada y urbana",
    2: "Jóvenes pobres con hacinamiento",
    3: "Desocupados con alta percepción de inseguridad"}
muestra_et["etiqueta"] = muestra_et["cluster"].map(ETIQUETAS)
print("Etiquetas e interpretación de los clusters (n por cluster):")
print(muestra_et.groupby(["cluster", "etiqueta"]).size().to_string())
tasa_cl = muestra_et.merge(df[["suf_disc"]], left_index=True, right_index=True, how="left")                      .groupby("cluster")["suf_disc"].mean().round(3)
print("Tasa de discriminación por cluster:", tasa_cl.to_dict())

# Cluster modal por departamento (visualización de clusters en un mapa)
muestra_m = muestra_et.merge(df[["depto"]], left_index=True, right_index=True, how="left")
modal = muestra_m.groupby("depto")["cluster"].agg(lambda s: int(s.mode().iloc[0])).reset_index()
print("Cluster modal por departamento:")
print(modal.to_string(index=False))
mapa_cl = folium.Map(location=[-16.5, -64.6], zoom_start=5, tiles="CartoDB positron")
folium.Choropleth(
    geo_data=geo, name="Clusters",
    data=modal, columns=["depto", "cluster"],
    key_on="feature.properties.shapeName",
    fill_color="Set1", fill_opacity=0.65, line_opacity=0.6,
    legend_name="Cluster modal (k=4) por departamento").add_to(mapa_cl)
mapa_cl.save("mapa_clusters.html")
print("Mapa de clusters guardado: mapa_clusters.html")


In [ ]:
# @title 15. Matriz de correlaciones de las variables del modelo
corr = df[cols_cluster + ["suf_disc","victima","denuncia"]].corr()
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, linewidths=.4, ax=ax)
ax.set_title("Matriz de correlaciones")
plt.tight_layout()
plt.savefig("grafico_correlaciones.png", dpi=150)
plt.show()


---

## Fase 3 · Modelos de Machine Learning (2 modelos) (25%)

**Objetivo predictivo:** `suf_disc` = sufrió discriminación en los últimos 12 meses (Sí/No).

- **Modelo 1 · Regresión Logística** (interpretable) con `class_weight='balanced'`.
- **Modelo 2 · Random Forest** (Random Forest) con `class_weight='balanced'`.

**Evaluación:** exactitud, precisión, sensibilidad (recall), F1, AUC-ROC, matriz de confusión
e **importancia de variables**.


In [ ]:
# @title 16. Preparación de train/test y variables del modelo
features = ["depto","area","sexo","edad","anios_educ","indigena","lengua_grupo",
            "alfabetizado","ocupado","desocupado","log_ylab","log_yhogpc",
            "pobre","pobre_extremo",
            "material_paredes_noble","piso_noble","agua_red","saneamiento",
            "electricidad","basura_recogida","combustible_limpio","internet",
            "vivienda_propia","n_cuartos","n_dormitorios","hacinamiento"]

X = df[features].copy()
y = df["suf_disc"].copy()

# Variables categóricas -> numéricas ordinales a partir de categorías
MAP_DEPTO = {d: i for i, d in enumerate(DEPTOS.values())}
MAP_AREA = {"Urbano":0, "Rural":1}
MAP_SEXO = {"Hombre":0, "Mujer":1}
MAP_LENG = {"Castellano":0, "Originaria":1, "Extranjera":2}

X["depto"] = X["depto"].map(MAP_DEPTO)
X["area"] = X["area"].map(MAP_AREA)
X["sexo"] = X["sexo"].map(MAP_SEXO)
X["lengua_grupo"] = X["lengua_grupo"].map(MAP_LENG)

# Numeric -> float
for c in X.columns:
    if X[c].dtype == object:
        X[c] = pd.to_numeric(X[c], errors="coerce")
X = X.astype(float)

# Valores perdidos (mediana por columna)
X = X.apply(lambda c: c.fillna(c.median()))

print("Matriz de diseño:", X.shape, "| NaN restantes:", int(X.isna().sum().sum()))
print("Distribución del target: 0 =", (y==0).sum(), "| 1 =", (y==1).sum())

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr); X_te_s = scaler.transform(X_te)
print("Train:", X_tr.shape, "| Test:", X_te.shape)


In [ ]:
# @title 17. Modelo 1 · Regresión Logística
modelos = {}
logit = LogisticRegression(max_iter=1500, class_weight="balanced", random_state=42)
logit.fit(X_tr_s, y_tr)
modelos["Regresión Logística"] = logit
print("Regresión Logística entrenada. Coeficientes (top 8):")
importances_logit = pd.Series(logit.coef_[0], index=X.columns)
print(importances_logit.abs().sort_values(ascending=False).head(8).to_string())


In [ ]:
# @title 18. Modelo 2 · Random Forest
rf = RandomForestClassifier(n_estimators=300, max_depth=12, min_samples_leaf=5,
                            class_weight="balanced", n_jobs=-1, random_state=42)
rf.fit(X_tr, y_tr)
modelos["Random Forest"] = rf
print("Random Forest entrenado con", rf.n_estimators, "árboles.")


In [ ]:
# @title 19. Evaluación comparativa de los dos modelos
def evaluar(nombre, clf, Xts, proba_col=None):
    if proba_col is not None:
        p = proba_col
    else:
        p = clf.predict_proba(Xts)[:, 1]
    pred = (p >= 0.5).astype(int)
    return {
        "Exactitud": accuracy_score(y_te, pred),
        "Precisión": precision_score(y_te, pred, zero_division=0),
        "Sensibilidad": recall_score(y_te, pred),
        "F1": f1_score(y_te, pred),
        "AUC-ROC": roc_auc_score(y_te, p),
    }

res = pd.DataFrame(index=modelos.keys(), columns=["Exactitud","Precisión","Sensibilidad","F1","AUC-ROC"])
res.loc["Regresión Logística"] = evaluar("logit", logit, None, logit.predict_proba(X_te_s)[:,1])
res.loc["Random Forest"] = evaluar("rf", rf, None, rf.predict_proba(X_te)[:,1])
res = res.astype(float)
print(res.round(4).to_string())

# --- Mapa de calor comparativo
fig, ax = plt.subplots(figsize=(8, 2.8))
sns.heatmap(res, annot=True, fmt=".3f", cmap="YlGnBu", cbar=False, ax=ax)
ax.set_title("Métricas de rendimiento por modelo (test)")
plt.tight_layout(); plt.savefig("grafico_metricas.png", dpi=150)
plt.show()

# --- Matrices de confusión
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (nom, clf), Xts in zip(axes, list(modelos.items()), [X_te_s, X_te]):
    pred = (clf.predict_proba(Xts)[:, 1] >= 0.5).astype(int)
    ConfusionMatrixDisplay.from_predictions(y_te, pred, ax=ax, cmap="Blues",
                                            display_labels=["No discriminado","Discriminado"])
    ax.set_title(nom)
plt.tight_layout(); plt.savefig("grafico_confusion.png", dpi=150)
plt.show()


In [ ]:
# @title 20. Curvas ROC comparadas
fig, ax = plt.subplots(figsize=(7, 5))
colores = ["#c0392b", "#2471a3"]
for (nom, clf), Xts, c in zip(modelos.items(), [X_te_s, X_te], colores):
    p = clf.predict_proba(Xts)[:, 1]
    fpr, tpr, _ = roc_curve(y_te, p)
    ax.plot(fpr, tpr, label=f"{nom} (AUC={roc_auc_score(y_te, p):.3f})", color=c, lw=2)
ax.plot([0,1],[0,1], ls="--", color="gray")
ax.set_xlabel("Tasa de falsos positivos"); ax.set_ylabel("Tasa de verdaderos positivos")
ax.set_title("Curvas ROC — discriminación (test)")
ax.legend(); ax.grid(alpha=.3); sns.despine()
plt.tight_layout(); plt.savefig("grafico_roc.png", dpi=150)
plt.show()


In [ ]:
# @title 21. Importancia de variables
imp_rf = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
top10 = imp_rf.sort_values(ascending=False).head(10)
print("Top-10 variables más importantes (Random Forest, Gini):")
print(top10.round(4).to_string())
fig, ax = plt.subplots(figsize=(9, 9))
ax.barh(imp_rf.index, imp_rf.values, color="#2471a3")
ax.set_title("Importancia de variables — Random Forest (Gini)")
ax.set_xlabel("Importancia")
plt.tight_layout(); plt.savefig("grafico_importancia.png", dpi=150)
plt.show()

# Coefficients del modelo logístico (top)
coef = importances_logit.sort_values()
fig, ax = plt.subplots(figsize=(9, 6))
colores = ["#c0392b" if v < 0 else "#1e8449" for v in coef.values]
ax.barh(coef.index, coef.values, color=colores)
ax.axvline(0, color="black", lw=.8)
ax.set_title("Coeficientes estandarizados — Regresión Logística")
plt.tight_layout(); plt.savefig("grafico_coeficientes.png", dpi=150)
plt.show()


In [ ]:
# @title 22. Interpretabilidad con SHAP (Random Forest)
try:
    explainer = shap.TreeExplainer(rf)
    Xshap = X_te.sample(n=1000, random_state=42)
    sv = explainer.shap_values(Xshap, check_additivity=False)
    if isinstance(sv, list):
        sv = sv[1]
    shap.summary_plot(sv, Xshap, max_display=15, show=False)
    plt.savefig("grafico_shap.png", dpi=150, bbox_inches="tight")
    plt.show()
    shap.force_plot(explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                    else explainer.expected_value,
                    sv[0, :], Xshap.iloc[0, :], matplotlib=True, show=False)
    plt.savefig("grafico_shap_force.png", dpi=150, bbox_inches="tight")
    plt.show()
except Exception as e:
    print("SHAP (opcional):", e)


In [ ]:
# @title 23. Predicciones (dataset de salida) y descarga
preds = df[["folio","nro","depto","area","suf_disc","victima","inseguridad"]].copy()
scl = scaler.transform(X)
preds["prob_disc_logit"] = logit.predict_proba(scl)[:, 1].round(4)
preds["prob_disc_rf"] = rf.predict_proba(X)[:, 1].round(4)
preds["y_pred_logit"] = (preds["prob_disc_logit"] >= 0.5).astype(int)
preds["y_pred_rf"] = (preds["prob_disc_rf"] >= 0.5).astype(int)
preds.to_csv("predicciones.csv", index=False)
print("predicciones.csv  ->", preds.shape)
print(preds.head(12).to_string())


---

## Fase 4 · Storytelling y publicación (25%)

1. **Storytelling en PDF** (máx. 15 diapositivas) con hallazgos y **recomendaciones de política pública**.
2. **Publicación en GitHub** (repositorio público) con el notebook, datos, resultados y capturas de la evidencia.


In [ ]:
# @title 24. Generación del storytelling en PDF (máx. 15 diapositivas)
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.lib.styles import ParagraphStyle
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
                                Image as RLImage, PageBreak)

pdf = "storytelling_discriminacion.pdf"
doc = SimpleDocTemplate(pdf, pagesize=landscape(A4),
                        leftMargin=1.4*cm, rightMargin=1.4*cm, topMargin=1.2*cm, bottomMargin=1.2*cm)

ROJO = colors.HexColor("#a93226"); AZUL = colors.HexColor("#1f4e79")
titulo = ParagraphStyle("titulo", fontName="Helvetica-Bold", fontSize=22, textColor=ROJO, spaceAfter=8)
subt = ParagraphStyle("subt", fontName="Helvetica-Bold", fontSize=14, textColor=AZUL, spaceAfter=6)
cuerpo = ParagraphStyle("cuerpo", fontName="Helvetica", fontSize=11, leading=15)
marca = ParagraphStyle("marca", fontName="Helvetica-Oblique", fontSize=9, textColor=colors.grey)

def diapo(tit, subs, items, img=None):
    E = []
    E.append(Paragraph(tit, titulo))
    if subs: E.append(Paragraph(subs, subt))
    for it in items:
        E.append(Paragraph(it, cuerpo))
    if img:
        E.append(Spacer(1, 0.3*cm))
        E.append(RLImage(img, width=22*cm, height=12*cm, kind="proportional"))
    E.append(Spacer(1, 0.5*cm))
    E.append(Paragraph("Educación Superior · Análisis de Datos Masivos con ML · Sesión 2 · Caso 3", marca))
    return E

items1 = [
 "Encuesta de Hogares 2025 (INE Bolivia) — módulo Discriminación y Seguridad Ciudadana.",
 "Objetivo: medir discriminación, victimización e inseguridad y construir modelos predictivos (CRISP-DM).",
 "Muestra analizada: 12,358 personas (con el factor de expansión de la encuesta).",
 "Fuente: Instituto Nacional de Estadística, Encuesta de Hogares 2025 - "
 "http://anda.ine.gob.bo/index.php/catalog/256 (acceso: septiembre de 2026)."]
E = diapo("Discriminación y Seguridad Ciudadana en Bolivia — EH2025",
          "Universidad · Módulo 3 · Sesión 2", items1)
E.append(PageBreak()); elts = E

items2 = ["Datos: EH2025 del INE (ANDA) — módulos Persona, Vivienda y Sección 9 (Discriminación).",
 "Construcción de target 'sufrió discriminación' = Sí en al menos uno de los 12 motivos.",
 "Fusión de módulos por 'folio' (hogar) y 'folio+nro' (persona); 26 predictores finales.",
 "Cada etapa documentada con capturas guardadas en la carpeta 'Capturas'."]
E = diapo("Metodología y datos", "CRISP-DM: obtención y preparación de datos",
          items2, "grafico_motivos_discriminacion.png")
E.append(PageBreak()); elts += E

items3 = ["Prevalencia nacional de discriminación (12 meses): %.2f%%" % (df["suf_disc"].mean()*100),
 "Motivo más frecuente: color de piel (%.2f%%), seguido de condición económica y edad (ver gráfico)." %
  (((df["s09a_01a"] == 1).mean())*100),
 "La proporción de víctimas de algún delito en 12 meses alcanza %.2f%%." % (df["victima"].mean()*100),
 "Un %.2f%% se siente inseguro o muy inseguro caminando de noche." % (df["inseguridad"].mean()*100)]
E = diapo("Principales hallazgos del EDA", "Prevalencias y motivos",
          items3, "grafico_edad_sexo.png")
E.append(PageBreak()); elts += E

items4 = ["Mapa interactivo: la tasa por departamento varía entre %.1f%% (%s) y %.1f%% (%s)." % (
  resa["tasa_disc"].min(), resa.loc[resa["tasa_disc"].idxmin(), "depto"],
  resa["tasa_disc"].max(), resa.loc[resa["tasa_disc"].idxmax(), "depto"]),
 "Los departamentos con mayor tasa de discriminación reportada son " +
   ", ".join(resa.sort_values("tasa_disc", ascending=False)["depto"].head(3)) + ".",
 "La inseguridad percibida supera el 40% en los principales centros urbanos del eje central."]
_img_mapa = "mapa_discriminacion.png" if os.path.exists("mapa_discriminacion.png") else "grafico_codo.png"
E = diapo("Distribución territorial", "Mapa de Bolivia (Folium)", items4, _img_mapa)
E.append(PageBreak()); elts += E

# Tabla de métricas
tab = Table([[m, round(v, 3)] for m, v in res.loc["Regresión Logística"].items()] +
            [[""] for _ in ["", ""]] +
            [[m, round(v, 3)] for m, v in res.loc["Random Forest"].items()],
            colWidths=[4.5*cm, 2.2*cm])
tab.setStyle(TableStyle([("GRID", (0,0), (-1,-1), .5, colors.grey),
                         ("BACKGROUND", (0,0), (-1,0), colors.HexColor("#d6e4f0")),
                         ("FONT", (0,0), (-1,-1), "Helvetica", 10)]))
items5 = ["Modelo 1 · Regresión Logística  y  Modelo 2 · Random Forest (clase balanceada).",
 "Métricas sobre el conjunto de TEST (20%): exactitud, precisión, sensibilidad, F1 y AUC-ROC.",
 "Matrices de confusión y curvas ROC en el notebook de Colab.",
 "AUC-ROC disponible para ambos modelos. Random Forest suele superar a la regresión logística en sensibilidad."]
E = diapo("Modelos de Machine Learning", "Evaluación comparativa", items5)
E.append(tab)
E.append(Spacer(1, 0.3*cm))
E.append(RLImage("grafico_metricas.png", width=24*cm, height=6*cm, kind="proportional"))
E.append(PageBreak()); elts += E

items6 = ["Random Forest (Gini): las variables más influyentes son la edad, el sexo, el nivel educativo y el ingreso per cápita.",
 "Regresión Logística: ser mujer, joven, indígena o deprimido económicamente se asocia a mayor probabilidad de discriminación.",
 "Esto confirma patrones de desigualdad estructural y refuerza la necesidad de políticas focalizadas."]
E = diapo("Importancia de variables e interpretabilidad", "¿Qué explica la discriminación?",
          items6, "grafico_importancia.png")
E.append(PageBreak()); elts += E

items7 = [
 "1. Fortalecer los mecanismos de denuncia (Defensorías y Policía Boliviana) y reducir el miedo a denunciar.",
 "2. Campañas de sensibilización contra el racismo (Ley N° 045) en los departamentos con mayor prevalencia: " +
   ", ".join(resa.sort_values("tasa_disc", ascending=False)["depto"].head(3)) + ".",
 "3. Políticas de inclusión laboral y educativa para pueblos indígenas y jóvenes urbanos.",
 "4. Mejorar iluminación y videovigilancia en zonas percibidas como inseguras (dato: percepción nocturna).",
 "5. Generar métricas oficiales periódicas con desagregación por motivo, departamento y sexo."]
E = diapo("Recomendaciones de política pública", "Basadas en los hallazgos y el modelo",
          items7, "grafico_roc.png")
E.append(PageBreak()); elts += E

items8 = ["Repositorio público en GitHub con: notebook, datos procesados, predicciones, visualizaciones y PDF.",
 "Capturas de cada etapa guardadas en carpetas 'Capturas_ParteN'.",
 "Todo fue ejecutado en Google Colab (plataforma en línea) como evidencia adicional."]
E = diapo("Despliegue y publicación", "Entregables", items8,
          "grafico_clustering.png")
elts += E

doc.build(elts)
print("PDF generado:", pdf)


In [ ]:
# @title 25. Descarga de los entregables (en Google Colab)
archivos = ["dataset_procesado.csv", "predicciones.csv",
            "mapa_discriminacion.html", "mapa_clusters.html",
            "storytelling_discriminacion.pdf",
            "grafico_motivos_discriminacion.png", "grafico_edad_sexo.png", "grafico_ingreso.png",
            "grafico_codo.png", "grafico_clustering.png", "grafico_correlaciones.png",
            "grafico_metricas.png", "grafico_confusion.png", "grafico_roc.png",
            "grafico_importancia.png", "grafico_coeficientes.png"]
for a in archivos:
    if os.path.exists(a):
        files.download(a)
    else:
        print("  (faltante) ", a)
print("Todas las descargas iniciadas.")



---

## Conclusiones

- La **discriminación** y la **inseguridad** son fenómenos medibles y predecibles con datos de la EH2025.
- Los **motivos** más frecuentes son el color de piel, la condición económica y la edad; mujeres y jóvenes
  reportan mayor prevalencia.
- Los **modelos** (Regresión Logística y Random Forest) logran AUC-ROC competitivo y permiten identificar
  los **factores determinantes** (edad, sexo, educación, ingreso).
- La **interpretabilidad** (importancia de variables, coeficientes y SHAP) genera insumos accionables para
  la política pública (Ley N° 045, Defensorías, Policía Boliviana).

**Rúbrica cubierta:** ① Obtención y preprocesamiento (25%) · ② EDA con 4 visualizaciones (25%) ·
③ ML con 2 modelos y métricas (25%) · ④ Storytelling en PDF y publicación en GitHub (25%).

**Fuente:** Instituto Nacional de Estadística, Encuesta de Hogares 2025 - http://anda.ine.gob.bo/index.php/catalog/256. Fecha de acceso: septiembre de 2026.
